# TD vs FD→TD mode validation

This notebook uses common parameters for `NRSur7dq4` and `IMRPhenomXPHM`, except `f_lower` (`0` for NRSur, `10 Hz` for the FD approximant). `phi_ref` is common. It then measures both a phase-only and a time+phase coalignment for `h22`.

All plot `xlim` values are in geometric units `(t-t_peak)/M`; `xlim=[...]` is accepted as before.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import lal
from waveformtools.models.lal import LALWaveformModel
from waveformtools.modes_array import ModesArray
from lalsimulation import SimInspiralGetApproximantFromString

TD_APPROXIMANT = "NRSur7dq4"
FD_APPROXIMANT = "IMRPhenomXPHM"
TD_F_LOWER = 0.0
FD_F_LOWER = 10.0
PHI_REF = 0.3
MODE = (2, 2)

COMMON = dict(
    mass1=35.0, mass2=30.0,
    spin1x=0.0, spin1y=0.0, spin1z=0.2,
    spin2x=0.0, spin2y=0.0, spin2z=-0.1,
    distance=400.0, inclination=0.7, phi_ref=PHI_REF,
    f_ref=20.0, f_max=512.0,
    delta_t=1/4096, delta_f=1/8, ell_max=4,
)

M_SECONDS = (COMMON["mass1"] + COMMON["mass2"]) * lal.MTSUN_SI
XLIM_PREPEAK = [-500.0, 0.0]
XLIM_NEAR_PEAK = [-500.0, 100.0]
XLIM_FULL = None
ALIGN_XLIM = [-300.0, -50.0]
TIME_SEARCH = [-150.0, 150.0]

print(f"M = {M_SECONDS:.9e} s")
print(f"common phi_ref = {PHI_REF}")


In [ ]:
def params(apx):
    p = dict(COMMON)
    p["approximant"] = apx
    if apx == TD_APPROXIMANT:
        p["f_lower"] = TD_F_LOWER
    elif apx == FD_APPROXIMANT:
        p["f_lower"] = FD_F_LOWER
    else:
        raise ValueError(apx)
    return p

def available(apx):
    try:
        SimInspiralGetApproximantFromString(apx)
        return True
    except Exception:
        return False

def tM(w, mode=MODE):
    h = np.asarray(w.mode(*mode))
    t = np.asarray(w.time_axis)
    if len(t) != len(h):
        return np.arange(len(h), dtype=float)
    return (t - t[np.argmax(np.abs(h))]) / M_SECONDS

def interp_complex(t0, h0, t1):
    idx = np.argsort(t0)
    t0 = np.asarray(t0)[idx]
    h0 = np.asarray(h0)[idx]
    return np.interp(t1, t0, h0.real) + 1j*np.interp(t1, t0, h0.imag)

def rel_l2(a, b, eps=1e-300):
    return np.linalg.norm(a-b) / max(np.linalg.norm(a), np.linalg.norm(b), eps)

def xlim_resolve(xlim=None, xlim_m=None):
    if xlim is not None and xlim_m is not None:
        raise ValueError("pass only one of xlim or xlim_m")
    return xlim if xlim is not None else xlim_m

def mode_series(w, phase=0.0, mode=MODE, normalize=False):
    h = np.asarray(w.mode(*mode)) * np.exp(1j*phase)
    if normalize:
        s = np.max(np.abs(h))
        h = h / (s if s > 0 and np.isfinite(s) else 1.0)
    return h


In [ ]:
def phase_for_tau(wref, wtest, tau=0.0, mode=MODE, xlim=ALIGN_XLIM):
    # Plot convention: aligned test has x_test = t_test + tau.
    # Thus at a reference time t, sample original test at t - tau.
    tr, tt = tM(wref, mode), tM(wtest, mode)
    hr, ht = np.asarray(wref.mode(*mode)), np.asarray(wtest.mode(*mode))
    lo, hi = xlim
    ts = tr[(tr >= lo) & (tr <= hi)]
    ts = ts[(ts - tau >= np.min(tt)) & (ts - tau <= np.max(tt))]
    if len(ts) < 8:
        return np.nan, np.inf, len(ts)
    hr_i = interp_complex(tr, hr, ts)
    ht_i = interp_complex(tt, ht, ts - tau)
    alpha = np.angle(np.vdot(ht_i, hr_i))
    return alpha, rel_l2(hr_i, ht_i*np.exp(1j*alpha)), len(ts)

def best_time_phase(wref, wtest, mode=MODE, xlim=ALIGN_XLIM, search=TIME_SEARCH):
    rows = []
    for tau in np.arange(search[0], search[1] + 0.5, 1.0):
        alpha, err, n = phase_for_tau(wref, wtest, tau, mode, xlim)
        rows.append((err, tau, alpha, n))
    _, tau0, _, _ = min(rows, key=lambda r: r[0])
    rows = []
    for tau in np.arange(tau0 - 1.0, tau0 + 1.0 + 0.025, 0.05):
        alpha, err, n = phase_for_tau(wref, wtest, tau, mode, xlim)
        rows.append((err, tau, alpha, n))
    err, tau, alpha, n = min(rows, key=lambda r: r[0])
    return tau, alpha, err, n

def phase_only(wref, wtest, mode=MODE, xlim=ALIGN_XLIM):
    tr, tt = tM(wref, mode), tM(wtest, mode)
    hr, ht = np.asarray(wref.mode(*mode)), np.asarray(wtest.mode(*mode))
    lo, hi = xlim
    ts = tr[(tr >= lo) & (tr <= hi)]
    ts = ts[(ts >= np.min(tt)) & (ts <= np.max(tt))]
    hr_i = interp_complex(tr, hr, ts)
    ht_i = interp_complex(tt, ht, ts)
    alpha = np.angle(np.vdot(ht_i, hr_i))
    return alpha, rel_l2(hr_i, ht_i), rel_l2(hr_i, ht_i*np.exp(1j*alpha)), len(ts)


In [ ]:
def unpack(item):
    if len(item) == 2:
        label, w = item; return label, w, 0.0, 0.0
    if len(item) == 3:
        label, w, ph = item; return label, w, ph, 0.0
    if len(item) == 4:
        label, w, ph, tau = item; return label, w, ph, tau
    raise ValueError("items are (label,w), (label,w,phase), or (label,w,phase,time_shift_M)")

def plot_re(items, title, mode=MODE, normalize=True, xlim=None, xlim_m=None):
    xl = xlim_resolve(xlim, xlim_m)
    plt.figure(figsize=(8,3.5))
    for item in items:
        label, w, ph, tau = unpack(item)
        plt.plot(tM(w, mode)+tau, mode_series(w, ph, mode, normalize).real, label=label)
    if xl is not None: plt.xlim(*xl)
    plt.xlabel(r"$(t-t_{\rm peak})/M$")
    plt.ylabel(f"Re h_{{{mode[0]}{mode[1]}}}" + (" / max|h|" if normalize else ""))
    plt.title(title); plt.legend(); plt.tight_layout()

def plot_amp(items, title, mode=MODE, normalize=True, xlim=None, xlim_m=None):
    xl = xlim_resolve(xlim, xlim_m)
    plt.figure(figsize=(8,3.5))
    for item in items:
        label, w, ph, tau = unpack(item)
        plt.plot(tM(w, mode)+tau, np.abs(mode_series(w, ph, mode, normalize)), label=label)
    if xl is not None: plt.xlim(*xl)
    plt.xlabel(r"$(t-t_{\rm peak})/M$")
    plt.ylabel(f"|h_{{{mode[0]}{mode[1]}}}|" + (" / max|h|" if normalize else ""))
    plt.title(title); plt.legend(); plt.tight_layout()


In [ ]:
tdp, fdp = params(TD_APPROXIMANT), params(FD_APPROXIMANT)
diffs = {k for k in sorted(set(tdp)|set(fdp)) if tdp.get(k) != fdp.get(k)}
print("Parameter differences:", diffs)
assert diffs == {"approximant", "f_lower"}

assert available(TD_APPROXIMANT), f"{TD_APPROXIMANT} unavailable"
assert available(FD_APPROXIMANT), f"{FD_APPROXIMANT} unavailable"

w_td = LALWaveformModel(parameters_dict=tdp).get_td_waveform_modes(dimensionless=False)
fd_model = LALWaveformModel(parameters_dict=fdp)
w_fd = fd_model.get_fd_waveform_modes(dimensionless=False)
w_fd_td = fd_model.get_fd_waveform_modes_as_td(dimensionless=False)
w_fd_td_alias = fd_model.get_fd_modes_as_td(dimensionless=False)
w_fd_auto_td = fd_model.get_td_waveform_modes(dimensionless=False)
w_fd_manual_td = w_fd.to_time_basis()

for w in [w_td, w_fd, w_fd_td, w_fd_td_alias, w_fd_auto_td, w_fd_manual_td]:
    assert isinstance(w, ModesArray)

print("TD t/M range:", tM(w_td)[[0,-1]])
print("FD→TD t/M range:", tM(w_fd_td)[[0,-1]])
print("FD route max |explicit-alias|:", np.max(np.abs(w_fd_td.mode(*MODE)-w_fd_td_alias.mode(*MODE))))
print("FD route max |explicit-auto| :", np.max(np.abs(w_fd_td.mode(*MODE)-w_fd_auto_td.mode(*MODE))))
print("FD route max |explicit-manual|:", np.max(np.abs(w_fd_td.mode(*MODE)-w_fd_manual_td.mode(*MODE))))


In [ ]:
phase_shift, err0, err_phase, n0 = phase_only(w_td, w_fd_td)
time_shift, time_phase_shift, err_time_phase, n1 = best_time_phase(w_td, w_fd_td)

print(f"Alignment window [M]: {ALIGN_XLIM}")
print(f"phase-only: alpha = {phase_shift:.12g} rad, samples = {n0}")
print(f"phase-only relative L2 before = {err0:.6e}")
print(f"phase-only relative L2 after  = {err_phase:.6e}")
print()
print(f"time+phase: time shift applied to FD axis = {time_shift:.12g} M, samples = {n1}")
print(f"time+phase: phase shift applied to FD h22  = {time_phase_shift:.12g} rad")
print(f"time+phase relative L2 after = {err_time_phase:.6e}")
print()
print("Approx orbital phase shift implied by final mode phase if h_lm -> exp(-i m phi) h_lm:")
print(f"delta_phi_ref ≈ {-time_phase_shift/MODE[1]:.12g} rad")


In [ ]:
plot_re([('NRSur7dq4', w_td),
         ('IMRPhenomXPHM FD→TD, unaligned', w_fd_td)],
        'Unaligned TD approximant vs FD→TD approximant: Re h22',
        xlim=XLIM_PREPEAK)

plot_re([('NRSur7dq4', w_td),
         ('IMRPhenomXPHM FD→TD, phase-only aligned', w_fd_td, phase_shift)],
        'Phase-only aligned TD approximant vs FD→TD approximant: Re h22',
        xlim=XLIM_PREPEAK)

plot_re([('NRSur7dq4', w_td),
         ('IMRPhenomXPHM FD→TD, time+phase aligned', w_fd_td, time_phase_shift, time_shift)],
        'Time+phase aligned TD approximant vs FD→TD approximant: Re h22',
        xlim=XLIM_PREPEAK)

plot_amp([('NRSur7dq4', w_td),
          ('IMRPhenomXPHM FD→TD, time shifted', w_fd_td, 0.0, time_shift)],
         'Time-aligned amplitude comparison: |h22|',
         xlim=XLIM_PREPEAK)


In [ ]:
fd_routes = [
    ('explicit get_fd_waveform_modes_as_td', w_fd_td),
    ('alias get_fd_modes_as_td', w_fd_td_alias),
    ('automatic get_td_waveform_modes', w_fd_auto_td),
    ('manual FD.to_time_basis', w_fd_manual_td),
]
plot_re(fd_routes, f'FD→TD route comparison: {FD_APPROXIMANT}', normalize=False, xlim=XLIM_FULL)
plot_amp(fd_routes, f'FD→TD route amplitude comparison: {FD_APPROXIMANT}', xlim=XLIM_FULL)
plot_re(fd_routes, f'FD→TD route comparison near peak: {FD_APPROXIMANT}', normalize=False, xlim=XLIM_NEAR_PEAK)
